# 04 — Composite model, portfolio backtest, and analysis

**What this does:** first evaluates the three composite weighting schemes, then constructs and backtests the configured portfolio, then produces performance, stability, factor, sector, style, cost, and concentration analysis.

**Run first:** notebook 01 must have produced valid Russell 3000 panel and factor files. Run notebook 03 first when a signal definition has changed.

**Produces:** research tables and charts in this notebook. It reads `data/panel_russell3000.parquet` and `data/factors.parquet` and writes no data or portfolio files.

**Time:** approximately 5–15 minutes. Each of the three major steps is intentionally separate so you can see whether the composite, construction, or attribution failed.

## Step 1 — Load the project and review model settings

Confirm the five blocks, equal composite weighting, portfolio band, 3% name cap, and 0.95–1.10 beta band before reading a backtest.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'config.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project folder containing config.py.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
import importlib
import config
config = importlib.reload(config)  # Refresh settings in a reused notebook kernel.
from src.notebook_support import (
    confirm_analysis,
    confirm_composite,
    confirm_portfolio,
    settings_table,
)

display(settings_table())
print('Backtest environment loaded. The notebook will not write portfolio or data files.')

## Step 2 — Build and evaluate the composite

This reproduces equal, inverse-volatility, and correlation-group weighting, then reports block risk contribution, leave-one-out results, and comparison with quality alone. Equal weighting remains the default unless a predeclared structural problem appears; it is never selected because it has the best historical Sharpe.

In [ ]:
from src.model.composite import run_composite_research

composite_research = run_composite_research()
display(confirm_composite(composite_research))
display(composite_research.risk_contribution)
display(composite_research.leave_one_out)
display(composite_research.figure)

## Step 3 — Construct and backtest the configured portfolio

This applies the common 10% entry and retention threshold, pre-weighting holdings guardrail, 0.02% minimum-position floor, benchmark weighting, name and sector caps, beta constraint, and size-based trading costs. The current baseline has 227 monthly returns through December 2025, about 13% net annualized return, about 17% volatility, and realized beta inside 0.95–1.10. Treat these as reasonableness ranges, not promises.

In [ ]:
from src.model.portfolio import run_portfolio_research

portfolio_backtest = run_portfolio_research()
display(confirm_portfolio(portfolio_backtest))
display(portfolio_backtest.annual_returns)

## Step 4 — Run the full portfolio analysis

This adds subperiod stability, Newey-West factor attribution, Brinson sector attribution, size and style profiles, block contribution, cost sensitivity, and charts. The confirmation highlights the limitations that are easiest to overlook: alpha significance, concentration in a few months, and top-ten weight.

In [ ]:
from src.model.portfolio_analysis import run_portfolio_analysis

analysis = run_portfolio_analysis()
display(confirm_analysis(analysis))
display(analysis.headline)
display(analysis.subperiod_headline)
display(analysis.sector_full_period)
display(analysis.cost_sensitivity)
for chart_name, figure in analysis.figures.items():
    print(chart_name.replace('_', ' ').title())
    display(figure)

## How to read the result

A successful run is not the same as a convincing model. Compare gross with net returns, first half with second half, equal- with value-weighted signal spreads, alpha with its t-statistic, and the portfolio with cheap factor alternatives. Read `docs/methodology.md` before using any headline number externally; it records the concentration, attribution, live-data, restatement, and universe limitations.

## What was produced and what comes next

This notebook produced composite diagnostics, the configured historical backtest, attribution tables, and presentation charts. It wrote no files. Permanent methodology and signal decisions live under `docs/`.

For a live target and trade list, use **02_monthly_update.ipynb**. For a company-level explanation, use **05_stock_lookup.ipynb**.

## If something fails

- **Missing panel or factors:** run notebook 01 and confirm both files.
- **Composite result variable is missing:** restart the kernel and run from Step 1.
- **Beta or holdings confirmation is outside its expected range:** stop and compare `config.py` with the approved settings before interpreting performance.
- **Memory or dead kernel:** close other notebooks and rerun this notebook alone.
- **Different headline result:** confirm panel cutoff, name cap, weighting scheme, beta constraint, and costs. Do not update documentation until the cause is understood. See `START_HERE.ipynb` for detailed troubleshooting.